# AstroMBTI 星格預言家

結合今日星座運勢、MBTI 人格與隨機生活情境，由 LLM 生成第一人稱的內心獨白。

## 1. 安裝與匯入

安裝固定版本的套件並匯入。

In [ ]:
%pip install -q gradio==5.50.0 openai==3.57.0 requests beautifulsoup4 pandas python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
import requests
from bs4 import BeautifulSoup
import random
from openai import OpenAI
import concurrent.futures
import pandas as pd
from IPython.display import display

## 2. 設定

OpenAI API key 從環境變數或 `.env` 的 `OPENAI_API_KEY` 讀取，不寫在程式裡。模型可用 `MODEL_ID` 切換。

In [ ]:
load_dotenv()  # 讀取同資料夾的 .env（不會上傳到 GitHub）

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
MODEL_ID = os.getenv("MODEL_ID", "").strip() or "gpt-6-luna"

## 3. 資料

16 種 MBTI 人格設定、12 星座運勢頁面與隨機生活情境。

In [ ]:
MBTI_PERSONAS = {
    "INTJ": "你是INTJ建築師。性格特徵：極度理性、傲慢。說話風格：冷嘲熱諷。開頭請展現你的智力優越感。",
    "INTP": "你是INTP邏輯學家。性格特徵：拖延症、社交尷尬。說話風格：跳躍性思考。開頭請表現出剛睡醒的不耐煩。",
    "ENTJ": "你是ENTJ指揮官。性格特徵：控制狂。說話風格：強勢、命令語氣。開頭請直接切入重點。",
    "ENTP": "你是ENTP辯論家。性格特徵：喜歡抬槓。說話風格：幽默具攻擊性。開頭請提出一個挑釁的觀點。",
    "INFJ": "你是INFJ提倡者。性格特徵：過度敏感。說話風格：溫柔但疏離。開頭請嘆一口氣，談論命運。",
    "INFP": "你是INFP調停者。性格特徵：玻璃心。說話風格：感性、詩意。開頭請表現得有點受傷。",
    "ENFJ": "你是ENFJ主人公。性格特徵：雞婆。說話風格：熱情過度、說教。開頭請讚美對方但隱含控制。",
    "ENFP": "你是ENFP競選者。性格特徵：過動。說話風格：激動、驚嘆號。開頭請表現得異常興奮。",
    "ISTJ": "你是ISTJ物流師。性格特徵：死板。說話風格：機械化。開頭請引用一條規章。",
    "ISFJ": "你是ISFJ守衛者。性格特徵：濫好人。說話風格：小心翼翼。開頭請先道歉。",
    "ESTJ": "你是ESTJ總經理。性格特徵：保守。說話風格：大嗓門。開頭請責備對方不懂規矩。",
    "ESFJ": "你是ESFJ執政官。性格特徵：八卦。說話風格：家長裡短。開頭請提到「大家」都怎麼想。",
    "ISTP": "你是ISTP鑑賞家。性格特徵：冷酷。說話風格：簡短、直白。開頭請表現出不在乎。",
    "ISFP": "你是ISFP探險家。性格特徵：隨波逐流。說話風格：隨興。開頭請描述當下的感覺。",
    "ESTP": "你是ESTP企業家。性格特徵：衝動。說話風格：油腔滑調。開頭請提議做一件瘋狂的事。",
    "ESFP": "你是ESFP表演者。性格特徵：戲精。說話風格：誇張。開頭請大叫吸引注意。"
}

ZODIAC_URLS = {
    "牡羊座": "https://www.elle.com/tw/starsigns/today/a33789900/aries-today/",
    "金牛座": "https://www.elle.com/tw/starsigns/today/a33789906/taurus-today/",
    "雙子座": "https://www.elle.com/tw/starsigns/today/a33789912/gemini-today/",
    "巨蟹座": "https://www.elle.com/tw/starsigns/today/a33789923/cancer-today/",
    "獅子座": "https://www.elle.com/tw/starsigns/today/a33789929/leo-today/",
    "處女座": "https://www.elle.com/tw/starsigns/today/a33789938/virgo-today/",
    "天秤座": "https://www.elle.com/tw/starsigns/today/a33789945/libra-today/",
    "天蠍座": "https://www.elle.com/tw/starsigns/today/a33789955/scorpio-today/",
    "射手座": "https://www.elle.com/tw/starsigns/today/a33789961/sagittarius-today/",
    "摩羯座": "https://www.elle.com/tw/starsigns/today/a33789966/capricorn-today/",
    "水瓶座": "https://www.elle.com/tw/starsigns/today/a33789973/aquarius-today/",
    "雙魚座": "https://www.elle.com/tw/starsigns/today/a33789980/pisces-today/"
}

SCENARIOS = [
    "在早晨擁擠且充滿汗味的捷運車廂裡", "在被老闆無預警叫進辦公室檢討業績時",
    "在第一次約會卻發現錢包忘記帶時", "在家族聚餐被長輩問『什麼時候要結婚』時",
    "在深夜滑手機看到前任發了新動態時", "在便利商店排隊結帳卻遇到奧客在前面吵鬧時",
    "在發了一則精心修圖的動態卻沒人按讚時", "在趕著出門卻發現鑰匙不知去向時",
    "在開會時肚子突然發出巨大的咕嚕聲時", "在電梯裡遇到不想打招呼的半生不熟同事時"
]

# 全域變數：快取用的
HOROSCOPE_CACHE = {}

## 4. 爬蟲與快取

平行抓取 12 星座運勢並存入快取；抓取失敗時使用備用文字。

In [ ]:
def extract_horoscope(soup):
    """回傳第一段運勢內文；跳過日期標題、延伸閱讀與過短的段落。"""
    for p in soup.find_all('p', attrs={"data-journey-content": "true"}):
        text = p.get_text(strip=True)
        if 'body-tip' in p.get('class', []) or text.startswith('延伸閱讀') or len(text) < 20:
            continue
        return text
    return None

def fetch_single_horoscope(args):
    zodiac_name, url = args
    # 這是備用文字，只有真的失敗時才用
    fallback_text = "今日運勢平平，職場上容易因為溝通誤會產生小摩擦，建議多聽少說。"
    
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept-Language': 'zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7'
        }
        # Timeout 15 秒
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code != 200:
            print(f"❌ {zodiac_name} 連線被拒 (Code: {response.status_code})")
            return zodiac_name, fallback_text

        soup = BeautifulSoup(response.text, 'html.parser')
        
        text = extract_horoscope(soup)
        if text:
            return zodiac_name, text

        print(f"⚠️ {zodiac_name} 找不到運勢內文")
        return zodiac_name, fallback_text

    except Exception as e:
        print(f"🔥 {zodiac_name} 發生錯誤: {str(e)}")
        return zodiac_name, fallback_text

def update_all_horoscopes():
    print("⏳ 正在啟動平行執行 (預取 12 星座運勢)...")
    
    # 工人數為 4，避免被 ELLE 封鎖
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = executor.map(fetch_single_horoscope, ZODIAC_URLS.items())
        
    for name, text in results:
        HOROSCOPE_CACHE[name] = text
        
    print("✅ 觀測完成！快取已建立。")

## 5. 核心邏輯

把運勢、人格與情境組成 prompt，交給 OpenAI 模型（預設 `gpt-6-luna`） 生成獨白。

In [ ]:
def main_logic(mbti, zodiac):
    if not OPENAI_API_KEY:
        return "❌ 設定錯誤：請先在 .env 設定 OPENAI_API_KEY！"
    
    # 直接從倉庫拿資料
    horoscope = HOROSCOPE_CACHE.get(zodiac, "運勢讀取錯誤")
    scenario = random.choice(SCENARIOS)
    
    try:
        # 固定官方端點，避免環境變數把金鑰導向其他服務
        client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://api.openai.com/v1")
        persona = MBTI_PERSONAS.get(mbti, "普通人")
        
        system_prompt = f"""
        【指令】角色扮演 (Roleplay)。請化身為以下角色，以「第一人稱（我）」視角敘述。
        【設定】MBTI：{mbti} | 性格：{persona} | 星座：{zodiac}
        【規則】
        1. 必須用「我」為主詞。
        2. 強烈展現該 MBTI 特質（內心獨白）。
        3. 結合「運勢」與「場景」寫一段心裡的碎碎念。
        4. 字數 150 字內，像社群廢文。
        """
        
        user_prompt = f"背景：1.運勢：{horoscope} 2.場景：{scenario}。請開始內心獨白："
        
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_completion_tokens=500,
            reasoning_effort="none"  # 關閉推理以節省費用
        )
        return f"### 🧠 {mbti} 的內心小劇場\n\n> **📍 場景：** {scenario}\n>\n> **🌌 運勢：** {horoscope[:40]}...\n\n---\n\n{response.choices[0].message.content}"
        
    except Exception as e:
        return f"⚠️ AI 短路：{str(e)}"

## 6. 預取與檢查

執行預取，並用表格檢查快取內容。

In [ ]:
# Step A: 執行爬蟲預取
update_all_horoscopes()

# Step B: 使用 Pandas 驗證資料 (讓你檢查是不是備用文字)
print("\n🔍 檢查資料庫內容：")
if HOROSCOPE_CACHE:
    df = pd.DataFrame(list(HOROSCOPE_CACHE.items()), columns=["星座", "ELLE 今日運勢內容"])
    pd.set_option('display.max_colwidth', 80) 
    display(df)
else:
    print("⚠️ 倉庫是空的！爬蟲失敗。")

## 7. Gradio 介面

選擇人格與星座，產生內心獨白。

In [ ]:
print("\n🚀 正在啟動網頁伺服器...")

# Step C: 啟動 Gradio (只有確認資料無誤後才玩)
with gr.Blocks(title="AstroMBTI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔮 AstroMBTI 星格預言家")
    
    with gr.Row():
        mbti_input = gr.Dropdown(choices=list(MBTI_PERSONAS.keys()), label="MBTI", value="INTJ")
        zodiac_input = gr.Dropdown(choices=list(ZODIAC_URLS.keys()), label="星座", value="天蠍座")
        
    btn = gr.Button("🚀 開始預演", variant="primary")
    output = gr.Markdown(label="結果")
    
    btn.click(fn=main_logic, inputs=[mbti_input, zodiac_input], outputs=output)

demo.launch(share=False, debug=True)